In [ ]:
import sys
import asyncio
import time
import os

import numpy as np

from lsst.ts import salobj

from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS

In [ ]:
print(os.environ["OSPL_URI"])
print(os.environ["LSST_DDS_PARTITION_PREFIX"])

In [ ]:
# This notebook is an attempt to track down the source 
# of the electrical noise which causes the horizontal banding in AuxTel images.

In [ ]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG

In [ ]:
#Start classes
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

In [ ]:
# Run 1 bias as a test
await latiss.take_bias(1)

In [ ]:
# Now run 50 biases, separated by 15 second delay
for i in range(50):
    await latiss.take_bias(1)
    await asyncio.sleep(15)

In [ ]:
# enable components, then take 50 more biases
await atcs.enable()
await latiss.enable()

In [ ]:
# Now run 50 biases, separated by 15 second delay
for i in range(50):
    await latiss.take_bias(1)
    await asyncio.sleep(15)

In [ ]:
# Ensure we're using Nasmyth 2
await atcs.rem.atptg.cmd_focusName.set_start(focus=3)

In [ ]:
current_az = 0.0
current_el = 80.0
current_rot = 10.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

In [ ]:
# Now run 50 biases, with variable delay between telescope az changes
for i in range(50):
    initial_delay = np.random.randint(1,14) # Delay between 1 and 14 seconds
    final delay = 15 - initial_delay
    shift = (2.0 * np.random.random() - 1.0) * 5.0 # Shift between +/- 5 degrees 
    await asyncio.sleep(initial_delay)
    await atcs.point_azel(current_az + shift, current_el, rot_tel=current_rot)
    await asyncio.sleep(final_delay)
    await latiss.take_bias(1)

In [ ]:
# Now run 50 biases, with variable delay between telescope el changes
for i in range(50):
    initial_delay = np.random.randint(1,14) # Delay between 1 and 14 seconds
    final delay = 15 - initial_delay
    shift = (2.0 * np.random.random() - 1.0) * 5.0 # Shift between +/- 5 degrees 
    await asyncio.sleep(initial_delay)
    await atcs.point_azel(current_az, current_el + shift, rot_tel=current_rot)
    await asyncio.sleep(final_delay)
    await latiss.take_bias(1)

In [ ]:
# Now run 50 biases, with variable delay between telescope rot changes
for i in range(50):
    initial_delay = np.random.randint(1,14) # Delay between 1 and 14 seconds
    final delay = 15 - initial_delay
    shift = (2.0 * np.random.random() - 1.0) * 5.0 # Shift between +/- 5 degrees 
    await asyncio.sleep(initial_delay)
    await atcs.point_azel(current_az, current_el, rot_tel=current_rot + shift)
    await asyncio.sleep(final_delay)
    await latiss.take_bias(1)

In [ ]:
# Now run 60 biases, with variable delay between telescope rot changes
# but moving through the whole range.  There was some indication in the on-sky tests
# that negative rot angles were worse.
current_rot = 165.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)
for i in range(56):
    initial_delay = np.random.randint(1,14) # Delay between 1 and 14 seconds
    final delay = 15 - initial_delay
    current_rot -= 6.0
    await asyncio.sleep(initial_delay)
    await atcs.point_azel(current_az, current_el, rot_tel=current_rot)
    await asyncio.sleep(final_delay)
    await latiss.take_bias(1)